# SECOM — what the data actually supports

This notebook is an **exploration**, not a training run. It answers one question:
*how much can a sensor-only model honestly claim about wafer lot failure?*

The short answer, established below from the data rather than asserted:
**not much, and the reason is the dataset, not the method.** That finding is why
YieldGuard's calibrated-confidence work moved to PHM 2016 CMP, where process
measurements and outcome are recorded on the same wafer.

Runs on Colab or locally. Every number printed is recomputed here.

In [ ]:
import io, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False})

UCI = "https://archive.ics.uci.edu/static/public/179/secom.zip"
LOCAL = Path("../data")          # repo cache, when run from src/models/tabular/colab/


def load_secom():
    """Repo cache first, then UCI. Returns (features, raw labels)."""
    if (LOCAL / "secom.data").exists():
        X = pd.read_csv(LOCAL / "secom.data", sep=" ", header=None)
        y = pd.read_csv(LOCAL / "secom_labels.data", sep=" ", header=None)[0]
        print("source: local repo cache")
        return X, y.to_numpy()

    import zipfile
    with urllib.request.urlopen(UCI, timeout=60) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    X = pd.read_csv(io.BytesIO(z.read("secom.data")), sep=" ", header=None)
    y = pd.read_csv(io.BytesIO(z.read("secom_labels.data")), sep=" ", header=None)[0]
    print("source: UCI archive")
    return X, y.to_numpy()


X_df, y_raw = load_secom()
X_df.columns = [f"sensor_{i}" for i in range(X_df.shape[1])]
y_raw = np.asarray(y_raw).ravel()
print(f"{X_df.shape[0]:,} lots x {X_df.shape[1]} sensors")

## 1 · The labels, and which sign means what

SECOM's convention is easy to get backwards, so derive it instead of trusting a
README. The rare class is the failure — that is the only assumption we need.

In [ ]:
vals, counts = np.unique(y_raw, return_counts=True)
FAIL = vals[np.argmin(counts)]
PASS = vals[np.argmax(counts)]
y = (y_raw == FAIL).astype(int)          # 1 = fail, from here on

for v, c in zip(vals, counts):
    role = "FAIL" if v == FAIL else "PASS"
    print(f"  label {v:+d} -> {role:4s}  {c:,} lots  ({c / len(y_raw):.1%})")
print(f"\nimbalance  1 : {counts.max() / counts.min():.1f}   (fail : pass)")

**1:14.** A model that predicts "pass" for every lot scores ~93.4% accuracy and
catches nothing. Accuracy is unusable here; everything below reports recall,
precision and PR-AUC on the fail class only.

## 2 · Missingness is not uniform

~4.5% of all cells are missing, which sounds mild. It is not spread evenly —
a handful of sensors are almost entirely absent while most are complete.

In [ ]:
na_rate = X_df.isna().mean()
print(f"overall missing cells      {X_df.isna().to_numpy().mean():.2%}")
print(f"sensors with any NaN       {(na_rate > 0).sum()} / {len(na_rate)}")
print(f"sensors  >80% NaN          {(na_rate > .80).sum()}   <- dropped entirely")
print(f"sensors  >50% NaN          {(na_rate > .50).sum()}")

fig, ax = plt.subplots(1, 2, figsize=(9, 2.8))
ax[0].hist(na_rate, bins=50, color="#7aa5c7")
ax[0].axvline(.80, color="#ca3b37", lw=1.2, ls="--")
ax[0].set(title="missing rate per sensor", xlabel="fraction NaN", ylabel="sensors",
          yscale="log")
top = na_rate.sort_values(ascending=False).head(12)[::-1]
ax[1].barh(top.index, top.values, color="#d98a87")
ax[1].set(title="worst 12 sensors", xlabel="fraction NaN")
ax[1].tick_params(labelsize=7)
fig.tight_layout(); plt.show()

### Why rows are imputed, never dropped

In [ ]:
complete = X_df.dropna()
kept_fail = y[complete.index].sum()
print(f"rows surviving a naive dropna()   {len(complete):,} / {len(X_df):,} "
      f"({len(complete)/len(X_df):.1%})")
print(f"fail rows surviving               {kept_fail} / {y.sum()}")
print("\n-> dropping rows discards the class we are trying to detect.")
print("   median imputation, fit on train only, is the defensible option.")

## 3 · Split first, then fit everything on train only

Imputation and scaling fit on all rows would leak validation statistics into
training. Order matters more than method here.

In [ ]:
Xtr_df, Xva_df, ytr, yva = train_test_split(
    X_df, y, test_size=.20, stratify=y, random_state=RANDOM_STATE)

keep = na_rate[na_rate <= .80].index          # threshold from train-independent NaN rate
med  = Xtr_df[keep].median()
Xtr  = Xtr_df[keep].fillna(med)
Xva  = Xva_df[keep].fillna(med)

scaler = StandardScaler().fit(Xtr)
Ztr, Zva = scaler.transform(Xtr), scaler.transform(Xva)

vt   = VarianceThreshold(0.05).fit(Ztr)
Ftr, Fva = vt.transform(Ztr), vt.transform(Zva)

print(f"sensors after >80% NaN drop     {Ztr.shape[1]}")
print(f"sensors after variance filter   {Ftr.shape[1]}  "
      f"({Ztr.shape[1] - Ftr.shape[1]} near-constant sensors removed)")
print(f"\nvalidation split   {len(yva)} lots = {(yva==0).sum()} pass + {yva.sum()} fail")
print(f"-> {yva.sum()} positives is the entire budget every metric below is scored on.")

## 4 · Can any single sensor separate pass from fail?

The most direct test available. Rank all sensors by univariate ROC-AUC on the
training rows. If the signal were strong, a few sensors would stand well clear
of 0.5.

In [ ]:
aucs = np.array([roc_auc_score(ytr, Ztr[:, j]) for j in range(Ztr.shape[1])])
two_sided = np.maximum(aucs, 1 - aucs)          # direction-agnostic
order = np.argsort(-two_sided)
names = np.array(keep)

print("strongest sensors, univariate (train):")
for j in order[:8]:
    print(f"  {names[j]:<12s} AUC {aucs[j]:.3f}   |separation| {two_sided[j]:.3f}")
print(f"\nbest single sensor   {two_sided[order[0]]:.3f}")
print(f"median sensor        {np.median(two_sided):.3f}   (0.500 = no information)")

### Is the best sensor better than chance, or just the best of 590 coin flips?

With 590 sensors, some will look good by luck. Permute the labels and take the
maximum each time — that gives the null distribution of "best of 590".

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
null_max = []
for _ in range(200):
    yp = rng.permutation(ytr)
    a  = np.array([roc_auc_score(yp, Ztr[:, j]) for j in order[:60]])
    null_max.append(np.max(np.maximum(a, 1 - a)))
null_max = np.array(null_max)

obs = two_sided[order[0]]
print(f"observed best-of-590 AUC      {obs:.3f}")
print(f"null best-of-60  95th pct     {np.percentile(null_max, 95):.3f}")
print(f"empirical p-value             {(null_max >= obs).mean():.3f}")

fig, ax = plt.subplots(figsize=(5, 2.4))
ax.hist(null_max, bins=25, color="#c8d4dd", label="label-permuted null")
ax.axvline(obs, color="#ca3b37", lw=1.5, label=f"observed {obs:.3f}")
ax.set(title="best single-sensor separation vs chance", xlabel="max |AUC|")
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

The best sensor **does** clear the null — the signal is real, not an artifact of
searching 590 columns. It is also small: AUC 0.681 on the strongest sensor, and a
median sensor sits at 0.520. Real but weak is the honest reading, and weak signal
spread thinly across hundreds of correlated columns is the hardest case for any
detector to exploit.

## 5 · The two classes are not geometrically separated

Project to two dimensions. If failures occupied their own region, an anomaly
detector would be the obviously right tool.

In [ ]:
P = PCA(n_components=2, random_state=RANDOM_STATE).fit(Ftr)
P_tr = P.transform(Ftr)

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].scatter(*P_tr[ytr == 0].T, s=6, c="#8fbfa4", alpha=.55, label="pass", lw=0)
ax[0].scatter(*P_tr[ytr == 1].T, s=18, c="#ca3b37", alpha=.9, label="fail", lw=0)
ax[0].set(title=f"PCA — {P.explained_variance_ratio_.sum():.1%} of variance",
          xlabel="PC1", ylabel="PC2")
ax[0].legend(fontsize=8)

# distance from the pass centroid: does failing mean "further out"?
centroid = Ftr[ytr == 0].mean(0)
d = np.linalg.norm(Ftr - centroid, axis=1)
ax[1].hist(d[ytr == 0], bins=45, density=True, color="#8fbfa4", alpha=.75, label="pass")
ax[1].hist(d[ytr == 1], bins=45, density=True, color="#ca3b37", alpha=.65, label="fail")
ax[1].set(title="distance from the passing centroid", xlabel="L2 distance in scaled space")
ax[1].legend(fontsize=8)
fig.tight_layout(); plt.show()

print(f"mean distance  pass {d[ytr==0].mean():7.2f}   fail {d[ytr==1].mean():7.2f}")
print(f"AUC of distance-as-a-detector   {roc_auc_score(ytr, d):.3f}")

## 6 · The typical failing lot is quieter than the typical passing lot

This is the single most important plot for reading any demo of this dataset.
Take the **medoid** of each class — its most typical member, not its loudest —
and compare peak deviation.

In [ ]:
def medoid(Z):
    c = Z.mean(0)
    return int(np.argmin(np.linalg.norm(Z - c, axis=1)))

m_fail, m_pass = medoid(Ztr[ytr == 1]), medoid(Ztr[ytr == 0])
v_fail, v_pass = Ztr[ytr == 1][m_fail], Ztr[ytr == 0][m_pass]

print(f"medoid FAIL lot — peak deviation  {np.abs(v_fail).max():.2f} sigma")
print(f"medoid PASS lot — peak deviation  {np.abs(v_pass).max():.2f} sigma")
loud = Ztr[ytr == 1][np.argmax(np.abs(Ztr[ytr == 1]).max(1))]
print(f"\nloudest FAIL lot in train         {np.abs(loud).max():.2f} sigma")
print("-> a demo built on the loudest row would look decisive and prove nothing.")

fig, ax = plt.subplots(figsize=(8, 2.6))
ax.plot(np.abs(v_pass), lw=.7, color="#3f8f6a", label="medoid PASS")
ax.plot(np.abs(v_fail), lw=.7, color="#ca3b37", label="medoid FAIL")
ax.axhline(3, color="#6a7d8c", ls="--", lw=.8)
ax.text(5, 3.15, "3 sigma", fontsize=7, color="#6a7d8c")
ax.set(title="per-sensor deviation, typical lot of each class",
       xlabel="sensor index", ylabel="|z|", ylim=(0, 6))
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

## 7 · What the shipped detector gets, and how uncertain that is

Isolation Forest trained on **passing rows only** — an unsupervised normality
model, matching `src/models/tabular/anomaly.py`.

In [ ]:
iso = IsolationForest(n_estimators=300, max_features=.7, contamination=.05,
                      random_state=RANDOM_STATE).fit(Ftr[ytr == 0])

cal = -iso.score_samples(Ftr[ytr == 0])         # calibrate on train PASS only
lo, hi = cal.min(), cal.max()
score = np.clip((-iso.score_samples(Fva) - lo) / (hi - lo), 0, 1)

thr  = 0.3733                                   # from checkpoints/model_meta.json
pred = (score >= thr).astype(int)
tp = int((pred & yva).sum()); fp = int((pred & ~yva.astype(bool)).sum())
fn = int(((1 - pred) & yva).sum()); tn = len(yva) - tp - fp - fn

print(f"                predicted pass   predicted fail")
print(f"  actual pass        {tn:4d}             {fp:3d}")
print(f"  actual fail        {fn:4d}             {tp:3d}")
print(f"\n  recall     {tp/(tp+fn):.3f}      ({tp} of {tp+fn} failures caught)")
print(f"  precision  {tp/max(tp+fp,1):.3f}      ({fp} false alarms)")
print(f"  ROC-AUC    {roc_auc_score(yva, score):.3f}")
print(f"  PR-AUC     {average_precision_score(yva, score):.3f}   "
      f"(baseline = prevalence {yva.mean():.3f})")

### A point estimate on 21 positives is close to meaningless

Bootstrap the validation set to see how wide the interval really is.

In [ ]:
boot = []
idx = np.arange(len(yva))
for _ in range(2000):
    b = rng.choice(idx, len(idx), replace=True)
    if yva[b].sum() == 0 or yva[b].sum() == len(b):
        continue
    boot.append(roc_auc_score(yva[b], score[b]))
boot = np.array(boot)
ci = np.percentile(boot, [2.5, 97.5])

print(f"ROC-AUC  {roc_auc_score(yva, score):.3f}   95% CI [{ci[0]:.3f}, {ci[1]:.3f}]")
print(f"fraction of resamples at or below chance:  {(boot <= .5).mean():.1%}")

fig, ax = plt.subplots(figsize=(5, 2.4))
ax.hist(boot, bins=40, color="#c8d4dd")
ax.axvline(.5, color="#ca3b37", lw=1.2, ls="--", label="chance")
ax.axvline(boot.mean(), color="#3b5062", lw=1.4, label="mean")
ax.set(title="bootstrapped ROC-AUC, 2000 resamples", xlabel="ROC-AUC")
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

**If the interval includes 0.5, the detector is not reliably better than chance
on this split.** We report it either way. A single number from 21 positives is
a headline, not a result.

## 8 · What this means for the product

| finding | consequence |
|---|---|
| 1:14 imbalance, 21 validation positives | accuracy unusable; ROC-AUC CI [0.44, 0.72] spans chance |
| best sensor AUC 0.681, median 0.520 | signal is real but weak — no smoking gun |
| classes overlap in PCA and in centroid distance | anomaly detection has little to separate |
| typical failing lot is *quieter* than a typical passing one | magnitude-based alarms will not work |
| sensors are anonymised | a flagged sensor cannot be mapped to a process step |

SECOM tells us **whether** a lot failed. It does not tell us what the sensors
measure, and it contains no wafer map, so no sensor reading in it can be shown
to explain a defect pattern. That is a property of the public data, not a gap
in the modelling.

### Where the calibrated claims come from instead

**PHM 2016 CMP** records process traces and the measured removal rate for the
*same wafer*. That link is measured, so a conformal interval over it has a
coverage guarantee that can be checked — and it is: 94.0% empirical coverage,
with excursion sensitivity moving from 34.7% (point estimate) to 93.9%
(interval). See `src/models/cmp/NOTES.md` and the `/prediction` screen.

SECOM stays in the product for what it can honestly support: **ranking which
sensors on a specific lot deviate most**, as one input among several, never as
a calibrated probability of failure.

In [ ]:
print("Reproduced from:")
print("  src/models/tabular/data_prep.py   split, imputation, scaling")
print("  src/models/tabular/anomaly.py     serving path and calibration")
print("  src/models/tabular/NOTES.md       recorded metrics and negative results")